# Genera

## Jupyter Hints
You must press Esc to enter Command Mode before using cell manipulation shortcuts.

To create a new cell below the currently selected one, press B. To create a cell above, press A. By default, newly inserted cells are code blocks. To change a selected cell into a text (Markdown) block, press M. To change it back to a code block, press Y. You can delete the currently selected cell by pressing D twice rapidly.

For execution, press Ctrl + Enter to run the cell and keep it selected. Press Shift + Enter to run the cell and automatically select the next one.

You must exicute a markdown block to see the text!

## Genera 
The two genera that have appeared in my work are L and $\hat A$.
Recall from Milnor-Stasheff $\S19$ or Hirzebruch the notion of a multiplicative sequence $K_n$. These are homogenous polynomials wuch that when summed they form a graded ring map (more or less). There is a bijection between these multiplicative sequences and formal power series, given by evaluating the multiplicative sequence (the sum of the terms) on $1+t$. Any such multiplicative sequence defines a genus, that is a map from manifolds to (say) the rationals which is zero for manifolds that are not $4k$ dimensional and otherwise $K(M^{4k}) = K_n(p_1, ..., p_k)[M]$ evaluating the polynomial on the Pontryagin classes of the manifold (of the tangent bundle) and then on the fundamental class of the manifold. Any such genera will descend to the cobordism ring. Because the Pontryagin numbers are complete invariants on the _rational_ cobordism ring it can be shown that any ring homomorphism from the rational cobordism ring corresponds to a genus.

The signature is such a ring map and therefore it has a genus that is called the L-genus. Its power series is given by the Taylor series of
$$\frac{\sqrt{t}}{\tanh \sqrt{t}}$$
 We want a program that will print the L polynomials.

Another genus is the A hat genus, which is given by the power series
$$\frac{\frac{1}{2}\sqrt{t}}{\sinh \frac{1}{2} \sqrt{t}}$$

The only other genus I have come accross is the Todd genus in Hirzebruchs book.

# Code
## Getting up to speed with the sage functions

In [86]:
#Imports
from sage.all import SymmetricFunctions, QQ

In [87]:
#Combinatorics
# Initialize the ring of symmetric functions over the rationals
Sym = SymmetricFunctions(QQ)
m = Sym.monomial()
e = Sym.e() # Initialize the elementary symmetric function basis

#Simple use case
n = 3 #Size of symetric group action (for me this is n => K_n)
for p in Partitions(n):
    print(f"For the partition {p} the monomial symetric polynomial is {m[p].expand(n,alphabet = 'x')}")

For the partition [3] the monomial symetric polynomial is x0^3 + x1^3 + x2^3
For the partition [2, 1] the monomial symetric polynomial is x0^2*x1 + x0*x1^2 + x0^2*x2 + x1^2*x2 + x0*x2^2 + x1*x2^2
For the partition [1, 1, 1] the monomial symetric polynomial is x0*x1*x2


In [88]:
#Power series coefficients
degree = 10 #How many coefficients do I need
x = var('x')

#Specify the power series that correspond to my genera
#Note that to get sage to find the power series I have made the substitution x = \sqrt(t),
#hence the coefficients I need are only those in even degree, the others should be zero
L_power = x/tanh(x)
A_power = ((1/2)*x) / sinh((1/2)*x)

print(f"The first {degree} coefficients of the L power series are given by {L_power.series(x,degree).list()[::2]}")
print(f"The first {degree} coefficients of the A power series are given by {A_power.series(x,degree).list()}")

The first 10 coefficients of the L power series are given by [1, 1/3, -1/45, 2/945, -1/4725]
The first 10 coefficients of the A power series are given by [1, 0, -1/24, 0, 7/5760, 0, -31/967680, 0, 127/154828800, 0]


## Defining a useful function

In [89]:
def genus(n, f, t):
    '''Return the K_n(p_1, ..., p_n) polynomial (multiplicative series) for a function f, in variable t'''
    R = PolynomialRing(QQ, names=[f'p{i}' for i in range(1, n + 1)])
    K_n = R.zero()
    
    Sym = SymmetricFunctions(QQ)
    m = Sym.monomial()
    e = Sym.e() # Initialize the elementary symmetric function basis
    
    # Use .coefficient() for safety, as .list() can truncate trailing zeros
    f_coefficients = [f.series(t, 2*n + 2).coefficient(t, 2*k) for k in range(n + 1)]

    for q in Partitions(n):
        lambda_I = prod(f_coefficients[i] for i in q)
        
        # 1. Convert the monomial symmetric function to the elementary basis
        m_in_e = e(m[q])
        
        # 2. Map e_k to the Pontryagin class p_k
        # m_in_e iterates as pairs of (partition mu, coefficient c)
        # A partition mu like [2, 1] represents the term e_2 * e_1
        q_poly = sum(c * prod(R.gen(k - 1) for k in mu) for mu, c in m_in_e)
        
        K_n += lambda_I * q_poly

    return K_n

# Computing Genera
The above can be skipped to not care how it works, and the below can be used to just see the outputs. 
Note that the power series that are used in genera are all even, that is the coefficients are only on the even powers of the variable. Another way of getting at the same thing is to compose with the square root function. To make sage compute the power series we have implimented the even power series, so **do not compose with the square root**.

Below is an example of how to use this to compute the polynomials for the $L$ genus and $\hat{A}$ genus, the code and results were verified against the values 1 - 5 in Milnor-Stasheff and Hurzebruch respectively. 

In [91]:
t = var('t')
L_power = t/tanh(t)
A_power = ((1/2)*t) / sinh((1/2)*t)

print(14175*genus(4, L_power, t))
print(genus(1, A_power, t))

-3*p1^4 + 22*p1^2*p2 - 19*p2^2 - 71*p1*p3 + 381*p4
-1/24*p1
